In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.signal import (
    find_peaks,
    butter,
    sosfiltfilt
)


# ============================================================
# USER SETTINGS
# ============================================================

root_folder = Path(r"D:\CV-Data_9_9_26")

sense_resistor_ohm = 99_200


# ============================================================
# CV SETTINGS
# ============================================================

# Example:
# 25 mHz = 0.025 Hz
cv_frequency_hz = 0.025


# Number of COMPLETE cycles to discard from beginning
#
# Example:
# discard_first_n_cycles = 1
#
discard_first_n_cycles = 0


# ============================================================
# CHANNEL 2 OFFSET CORRECTION
# ============================================================

# Once measured at your chosen vertical scale:
#
# apply_ch2_offset_correction = True
# ch2_offset_mV = 2.7
#
apply_ch2_offset_correction = True

ch2_offset_mV = 5.3


# ============================================================
# LOW-PASS FILTER
# ============================================================

# Filter is applied to corrected Ch2 before current calculation.
#
# Raw and filtered signals are BOTH saved.
#
apply_lowpass_filter = True

# For a 25 mHz CV, 1 Hz is already 40x above
# the fundamental frequency.
lowpass_cutoff_hz = 1.0

lowpass_order = 4


# ============================================================
# MEAN CV PLOT
# ============================================================

# Number of points used when interpolating individual sweeps
# onto a common voltage grid.
mean_cv_points = 500


# ============================================================
# LOAD TEKTRONIX CSV
# ============================================================

def load_tektronix_csv(filename):

    df = pd.read_csv(
        filename,
        header=None,
        usecols=[3, 4],
        names=["time_s", "voltage_v"]
    )

    df["time_s"] = pd.to_numeric(
        df["time_s"],
        errors="coerce"
    )

    df["voltage_v"] = pd.to_numeric(
        df["voltage_v"],
        errors="coerce"
    )

    df = df.dropna().reset_index(drop=True)

    return df


# ============================================================
# FIND CHANNEL FILES
# ============================================================

def find_channel_files(folder):

    csv_files = list(folder.glob("*.csv"))

    ch1_candidates = []
    ch2_candidates = []

    for file in csv_files:

        name = file.stem.lower()

        if name.endswith("ch1") or name.endswith("_1"):
            ch1_candidates.append(file)

        elif name.endswith("ch2") or name.endswith("_2"):
            ch2_candidates.append(file)

    if len(ch1_candidates) != 1:

        raise ValueError(
            f"{folder.name}: expected exactly one Ch1 file, "
            f"found {len(ch1_candidates)}"
        )

    if len(ch2_candidates) != 1:

        raise ValueError(
            f"{folder.name}: expected exactly one Ch2 file, "
            f"found {len(ch2_candidates)}"
        )

    return (
        ch1_candidates[0],
        ch2_candidates[0]
    )


# ============================================================
# ALIGN CHANNELS
# ============================================================

def align_channels(ch1_df, ch2_df):

    n = min(
        len(ch1_df),
        len(ch2_df)
    )

    ch1_df = ch1_df.iloc[:n].copy()
    ch2_df = ch2_df.iloc[:n].copy()

    time1 = ch1_df["time_s"].to_numpy()
    time2 = ch2_df["time_s"].to_numpy()

    voltage1 = ch1_df["voltage_v"].to_numpy()
    voltage2 = ch2_df["voltage_v"].to_numpy()

    dt = np.median(
        np.diff(time1)
    )

    max_time_difference = np.max(
        np.abs(
            time1 - time2
        )
    )

    if max_time_difference > 0.1 * dt:

        print(
            "  WARNING: Ch1 and Ch2 timestamps differ "
            "by >10% of one sample interval."
        )

    return (
        time1,
        voltage1,
        voltage2
    )


# ============================================================
# PARSE ELECTRODE NAME
# ============================================================

def parse_electrode_name(folder_name):

    # Example:
    #
    # 4_1_Al_150
    #
    # interpreted as:
    #
    # 150 x 150 um square electrode

    parts = folder_name.split("_")

    if len(parts) < 4:

        raise ValueError(
            f"Folder '{folder_name}' does not match "
            "Wafer_Device_Material_Size"
        )

    size_um = float(
        parts[3]
    )

    return {

        "Wafer":
            parts[0],

        "Device":
            parts[1],

        "Material":
            parts[2],

        "Size_um":
            size_um
    }


# ============================================================
# ELECTRODE AREA
# ============================================================

def calculate_electrode_area(size_um):

    area_um2 = (
        size_um ** 2
    )

    # 1 cm² = 1e8 um²
    area_cm2 = (
        area_um2 / 1e8
    )

    return (
        area_um2,
        area_cm2
    )


# ============================================================
# LOW-PASS FILTER
# ============================================================

def lowpass_filter(
    signal,
    time_s,
    cutoff_hz,
    order
):

    dt = np.median(
        np.diff(time_s)
    )

    fs = 1 / dt

    nyquist_hz = (
        fs / 2
    )

    if cutoff_hz >= nyquist_hz:

        raise ValueError(
            f"Low-pass cutoff ({cutoff_hz} Hz) "
            f"must be below Nyquist frequency "
            f"({nyquist_hz:.3f} Hz)."
        )

    sos = butter(
        order,
        cutoff_hz,
        btype="low",
        fs=fs,
        output="sos"
    )

    filtered_signal = (
        sosfiltfilt(
            sos,
            signal
        )
    )

    return filtered_signal


# ============================================================
# MOVING AVERAGE
#
# Used ONLY for turning-point detection.
# It is NOT used for CSC integration.
# ============================================================

def moving_average(
    x,
    window
):

    if window <= 1:
        return x.copy()

    kernel = (
        np.ones(window)
        /
        window
    )

    return np.convolve(
        x,
        kernel,
        mode="same"
    )


# ============================================================
# DETECT COMPLETE CV CYCLES
# ============================================================

def detect_cycles(
    time_s,
    cell_voltage_v
):

    """
    Complete cycle:

        voltage minimum
            ->
        voltage maximum
            ->
        next voltage minimum

    Partial cycles at beginning/end are ignored.
    """

    dt = np.median(
        np.diff(time_s)
    )

    expected_period_s = (
        1 / cv_frequency_hz
    )

    expected_period_samples = int(
        expected_period_s / dt
    )


    # --------------------------------------------------------
    # Smooth voltage ONLY for turning-point detection
    # --------------------------------------------------------

    smoothing_window = max(
        3,
        int(
            expected_period_samples
            *
            0.005
        )
    )

    if smoothing_window % 2 == 0:
        smoothing_window += 1


    smoothed_voltage = (
        moving_average(
            cell_voltage_v,
            smoothing_window
        )
    )


    voltage_range = (
        np.max(smoothed_voltage)
        -
        np.min(smoothed_voltage)
    )


    # Extrema must be significant relative to overall range
    prominence = (
        0.20
        *
        voltage_range
    )


    minimum_peak_distance = max(
        1,
        int(
            expected_period_samples
            *
            0.60
        )
    )


    # --------------------------------------------------------
    # Find minima
    # --------------------------------------------------------

    minima, _ = find_peaks(

        -smoothed_voltage,

        prominence=
            prominence,

        distance=
            minimum_peak_distance
    )


    # --------------------------------------------------------
    # Find maxima
    # --------------------------------------------------------

    maxima, _ = find_peaks(

        smoothed_voltage,

        prominence=
            prominence,

        distance=
            minimum_peak_distance
    )


    print(
        f"  Detected minima: "
        f"{len(minima)}"
    )

    print(
        f"  Detected maxima: "
        f"{len(maxima)}"
    )


    # --------------------------------------------------------
    # Construct validated cycles
    # --------------------------------------------------------

    cycles = []


    for i in range(
        len(minima) - 1
    ):

        start = minima[i]

        end = minima[i + 1]


        maxima_inside = maxima[
            (maxima > start)
            &
            (maxima < end)
        ]


        if len(maxima_inside) == 0:
            continue


        # Pick largest maximum inside cycle
        max_index = maxima_inside[
            np.argmax(
                cell_voltage_v[
                    maxima_inside
                ]
            )
        ]


        cycle_duration_s = (
            time_s[end]
            -
            time_s[start]
        )


        # Allow +/-30% tolerance
        if not (

            0.70
            *
            expected_period_s

            <=

            cycle_duration_s

            <=

            1.30
            *
            expected_period_s

        ):

            continue


        cycles.append({

            "start":
                start,

            "maximum":
                max_index,

            "end":
                end
        })


    return cycles


# ============================================================
# CALCULATE BASE CV SIGNALS
# ============================================================

def calculate_cv_arrays(
    time_s,
    total_voltage_v,
    raw_sense_voltage_v,
    area_cm2
):

    # --------------------------------------------------------
    # Offset correction
    # --------------------------------------------------------

    if apply_ch2_offset_correction:

        offset_v = (
            ch2_offset_mV
            /
            1000
        )

    else:

        offset_v = 0.0


    corrected_sense_voltage_raw_v = (
        raw_sense_voltage_v
        -
        offset_v
    )


    # --------------------------------------------------------
    # Low-pass filter
    # --------------------------------------------------------

    if apply_lowpass_filter:

        corrected_sense_voltage_filtered_v = (
            lowpass_filter(

                corrected_sense_voltage_raw_v,

                time_s,

                cutoff_hz=
                    lowpass_cutoff_hz,

                order=
                    lowpass_order
            )
        )

    else:

        corrected_sense_voltage_filtered_v = (
            corrected_sense_voltage_raw_v.copy()
        )


    # --------------------------------------------------------
    # RAW CURRENT
    # --------------------------------------------------------

    current_raw_a = (
        corrected_sense_voltage_raw_v
        /
        sense_resistor_ohm
    )


    # --------------------------------------------------------
    # FILTERED CURRENT
    #
    # THIS is used for CSC.
    # --------------------------------------------------------

    current_filtered_a = (
        corrected_sense_voltage_filtered_v
        /
        sense_resistor_ohm
    )


    # --------------------------------------------------------
    # CELL VOLTAGE
    #
    # Use offset-corrected Ch2.
    #
    # Filtering Ch2 makes virtually no difference to cell
    # voltage because the sense voltage is small, but we keep
    # the filtered version consistent with current.
    # --------------------------------------------------------

    cell_voltage_v = (
        total_voltage_v
        -
        corrected_sense_voltage_filtered_v
    )


    # --------------------------------------------------------
    # CURRENT DENSITY
    # --------------------------------------------------------

    current_density_raw_mA_cm2 = (

        current_raw_a
        /
        area_cm2
        *
        1000
    )


    current_density_filtered_mA_cm2 = (

        current_filtered_a
        /
        area_cm2
        *
        1000
    )


    return (

        corrected_sense_voltage_raw_v,

        corrected_sense_voltage_filtered_v,

        cell_voltage_v,

        current_raw_a,

        current_filtered_a,

        current_density_raw_mA_cm2,

        current_density_filtered_mA_cm2
    )


# ============================================================
# ANALYZE INDIVIDUAL CYCLES
# ============================================================

def analyze_cycles(
    time_s,
    cell_voltage_v,
    current_a,
    area_cm2
):

    cycles = detect_cycles(
        time_s,
        cell_voltage_v
    )


    if len(cycles) == 0:

        raise ValueError(
            "No complete CV cycles were detected."
        )


    print(
        f"  Complete cycles detected: "
        f"{len(cycles)}"
    )


    retained_cycles = cycles[
        discard_first_n_cycles:
    ]


    if len(retained_cycles) == 0:

        raise ValueError(
            "No cycles remain after discarding "
            "conditioning cycles."
        )


    cycle_rows = []


    for cycle_number, cycle in enumerate(

        retained_cycles,

        start=1
    ):

        start = cycle["start"]

        maximum = cycle["maximum"]

        end = cycle["end"]


        sl = slice(
            start,
            end + 1
        )


        t = time_s[sl]

        v = cell_voltage_v[sl]

        i = current_a[sl]


        # ----------------------------------------------------
        # Positive current = anodic
        # Negative current = cathodic
        # ----------------------------------------------------

        anodic_current = np.maximum(
            i,
            0
        )

        cathodic_current = np.minimum(
            i,
            0
        )


        # ----------------------------------------------------
        # Charge
        # Q = integral I dt
        # ----------------------------------------------------

        anodic_charge_c = (
            np.trapezoid(
                anodic_current,
                t
            )
        )


        cathodic_charge_signed_c = (
            np.trapezoid(
                cathodic_current,
                t
            )
        )


        cathodic_charge_c = abs(
            cathodic_charge_signed_c
        )


        # ----------------------------------------------------
        # CSC
        # ----------------------------------------------------

        anodic_csc_mC_cm2 = (

            anodic_charge_c
            /
            area_cm2
            *
            1000
        )


        cathodic_csc_mC_cm2 = (

            cathodic_charge_c
            /
            area_cm2
            *
            1000
        )


        duration_s = (
            t[-1]
            -
            t[0]
        )


        dv_dt = np.gradient(
            v,
            t
        )


        mean_scan_rate_mV_s = (

            np.mean(
                np.abs(dv_dt)
            )

            *
            1000
        )


        cycle_rows.append({

            "Cycle":
                cycle_number,

            "Start_Index":
                start,

            "Maximum_Index":
                maximum,

            "End_Index":
                end,

            "Start_Time_s":
                time_s[start],

            "End_Time_s":
                time_s[end],

            "Duration_s":
                duration_s,

            "V_Min_V":
                np.min(v),

            "V_Max_V":
                np.max(v),

            "Mean_Abs_Scan_Rate_mV_s":
                mean_scan_rate_mV_s,

            "Cathodic_Charge_uC":
                cathodic_charge_c
                *
                1e6,

            "Anodic_Charge_uC":
                anodic_charge_c
                *
                1e6,

            "Cathodic_CSC_mC_cm2":
                cathodic_csc_mC_cm2,

            "Anodic_CSC_mC_cm2":
                anodic_csc_mC_cm2
        })


    return (
        pd.DataFrame(cycle_rows),
        retained_cycles
    )


# ============================================================
# PLOT INDIVIDUAL OVERLAID CYCLES
# ============================================================

def plot_overlaid_cycles(
    folder,
    time_s,
    cell_voltage_v,
    current_density_mA_cm2,
    retained_cycles
):

    plt.figure(
        figsize=(7, 6)
    )


    for cycle_number, cycle in enumerate(
        retained_cycles,
        start=1
    ):

        start = cycle["start"]
        end = cycle["end"]

        sl = slice(
            start,
            end + 1
        )

        plt.plot(
            cell_voltage_v[sl],
            current_density_mA_cm2[sl],
            label=f"Cycle {cycle_number}"
        )


    plt.axhline(
        0,
        linewidth=0.8
    )


    plt.xlabel(
        "Cell Voltage (V)"
    )

    plt.ylabel(
        "Current Density (mA/cm²)"
    )

    plt.title(
        f"CV Cycles - {folder.name}"
    )


    if len(retained_cycles) <= 10:

        plt.legend()


    plt.tight_layout()


    plt.savefig(

        folder /
        f"{folder.name}_CV_cycles_overlay.png",

        dpi=300
    )


    plt.close()


# ============================================================
# CREATE MEAN CV
#
# The increasing and decreasing sweeps must be averaged
# separately. Otherwise one voltage can correspond to two
# different currents within the CV loop.
# ============================================================

def calculate_mean_cv(
    cell_voltage_v,
    current_density_mA_cm2,
    retained_cycles
):

    forward_sweeps = []
    reverse_sweeps = []


    # --------------------------------------------------------
    # Determine common usable voltage range
    # --------------------------------------------------------

    forward_min_values = []
    forward_max_values = []

    reverse_min_values = []
    reverse_max_values = []


    for cycle in retained_cycles:

        start = cycle["start"]

        maximum = cycle["maximum"]

        end = cycle["end"]


        # Increasing-voltage branch
        vf = cell_voltage_v[
            start:
            maximum + 1
        ]

        jf = current_density_mA_cm2[
            start:
            maximum + 1
        ]


        # Decreasing-voltage branch
        vr = cell_voltage_v[
            maximum:
            end + 1
        ]

        jr = current_density_mA_cm2[
            maximum:
            end + 1
        ]


        forward_sweeps.append(
            (vf, jf)
        )

        reverse_sweeps.append(
            (vr, jr)
        )


        forward_min_values.append(
            np.min(vf)
        )

        forward_max_values.append(
            np.max(vf)
        )

        reverse_min_values.append(
            np.min(vr)
        )

        reverse_max_values.append(
            np.max(vr)
        )


    # Voltage range common to ALL cycles
    forward_v_min = max(
        forward_min_values
    )

    forward_v_max = min(
        forward_max_values
    )


    reverse_v_min = max(
        reverse_min_values
    )

    reverse_v_max = min(
        reverse_max_values
    )


    forward_grid = np.linspace(
        forward_v_min,
        forward_v_max,
        mean_cv_points
    )


    # For reverse branch we want plotting direction
    # high voltage -> low voltage
    reverse_grid_ascending = np.linspace(
        reverse_v_min,
        reverse_v_max,
        mean_cv_points
    )


    # --------------------------------------------------------
    # Interpolate each forward sweep
    # --------------------------------------------------------

    forward_interpolated = []


    for v, j in forward_sweeps:

        # np.interp requires ascending x values
        order = np.argsort(v)

        v_sorted = v[order]
        j_sorted = j[order]


        # Remove duplicate voltages if needed
        v_unique, unique_indices = np.unique(
            v_sorted,
            return_index=True
        )

        j_unique = j_sorted[
            unique_indices
        ]


        interp_j = np.interp(
            forward_grid,
            v_unique,
            j_unique
        )

        forward_interpolated.append(
            interp_j
        )


    # --------------------------------------------------------
    # Interpolate each reverse sweep
    # --------------------------------------------------------

    reverse_interpolated = []


    for v, j in reverse_sweeps:

        order = np.argsort(v)

        v_sorted = v[order]
        j_sorted = j[order]


        v_unique, unique_indices = np.unique(
            v_sorted,
            return_index=True
        )

        j_unique = j_sorted[
            unique_indices
        ]


        interp_j = np.interp(
            reverse_grid_ascending,
            v_unique,
            j_unique
        )

        reverse_interpolated.append(
            interp_j
        )


    forward_interpolated = np.array(
        forward_interpolated
    )

    reverse_interpolated = np.array(
        reverse_interpolated
    )


    # --------------------------------------------------------
    # Mean and SD
    # --------------------------------------------------------

    forward_mean = np.mean(
        forward_interpolated,
        axis=0
    )

    reverse_mean = np.mean(
        reverse_interpolated,
        axis=0
    )


    if len(retained_cycles) > 1:

        forward_sd = np.std(
            forward_interpolated,
            axis=0,
            ddof=1
        )

        reverse_sd = np.std(
            reverse_interpolated,
            axis=0,
            ddof=1
        )

    else:

        forward_sd = np.zeros_like(
            forward_mean
        )

        reverse_sd = np.zeros_like(
            reverse_mean
        )


    # Reverse plotting direction
    reverse_grid = (
        reverse_grid_ascending[::-1]
    )

    reverse_mean = (
        reverse_mean[::-1]
    )

    reverse_sd = (
        reverse_sd[::-1]
    )


    return (

        forward_grid,
        forward_mean,
        forward_sd,

        reverse_grid,
        reverse_mean,
        reverse_sd
    )


# ============================================================
# PLOT MEAN CV
# ============================================================

def plot_mean_cv(
    folder,
    cell_voltage_v,
    current_density_mA_cm2,
    retained_cycles
):

    (

        forward_v,
        forward_mean,
        forward_sd,

        reverse_v,
        reverse_mean,
        reverse_sd

    ) = calculate_mean_cv(

        cell_voltage_v,

        current_density_mA_cm2,

        retained_cycles
    )


    plt.figure(
        figsize=(7, 6)
    )


    # --------------------------------------------------------
    # Forward sweep
    # --------------------------------------------------------

    plt.plot(
        forward_v,
        forward_mean,
        label="Increasing voltage"
    )


    plt.fill_between(

        forward_v,

        forward_mean
        -
        forward_sd,

        forward_mean
        +
        forward_sd,

        alpha=0.2
    )


    # --------------------------------------------------------
    # Reverse sweep
    # --------------------------------------------------------

    plt.plot(
        reverse_v,
        reverse_mean,
        label="Decreasing voltage"
    )


    plt.fill_between(

        reverse_v,

        reverse_mean
        -
        reverse_sd,

        reverse_mean
        +
        reverse_sd,

        alpha=0.2
    )


    plt.axhline(
        0,
        linewidth=0.8
    )


    plt.xlabel(
        "Cell Voltage (V)"
    )

    plt.ylabel(
        "Current Density (mA/cm²)"
    )

    plt.title(
        f"Mean CV - {folder.name}"
    )


    plt.legend()


    plt.tight_layout()


    plt.savefig(

        folder /
        f"{folder.name}_CV_mean_SD.png",

        dpi=300
    )


    plt.close()


    # --------------------------------------------------------
    # Also return mean data so it can be saved as CSV
    # --------------------------------------------------------

    forward_df = pd.DataFrame({

        "Scan_Direction":
            "Increasing",

        "Cell_Voltage_V":
            forward_v,

        "Mean_Current_Density_mA_cm2":
            forward_mean,

        "SD_Current_Density_mA_cm2":
            forward_sd
    })


    reverse_df = pd.DataFrame({

        "Scan_Direction":
            "Decreasing",

        "Cell_Voltage_V":
            reverse_v,

        "Mean_Current_Density_mA_cm2":
            reverse_mean,

        "SD_Current_Density_mA_cm2":
            reverse_sd
    })


    mean_df = pd.concat(
        [
            forward_df,
            reverse_df
        ],
        ignore_index=True
    )


    mean_df.to_csv(

        folder /
        f"{folder.name}_CV_mean_SD.csv",

        index=False
    )


# ============================================================
# PROCESS ONE ELECTRODE
# ============================================================

def process_electrode(folder):

    print()
    print("=" * 60)
    print(f"Processing: {folder.name}")
    print("=" * 60)


    # --------------------------------------------------------
    # Metadata
    # --------------------------------------------------------

    info = parse_electrode_name(
        folder.name
    )

    size_um = (
        info["Size_um"]
    )


    (
        area_um2,
        area_cm2

    ) = calculate_electrode_area(
        size_um
    )


    print(
        f"  Electrode: "
        f"{size_um:g} x {size_um:g} um"
    )

    print(
        f"  Area: "
        f"{area_um2:g} um²"
    )


    # --------------------------------------------------------
    # Find files
    # --------------------------------------------------------

    (
        ch1_file,
        ch2_file

    ) = find_channel_files(
        folder
    )


    print(
        f"  Ch1: "
        f"{ch1_file.name}"
    )

    print(
        f"  Ch2: "
        f"{ch2_file.name}"
    )


    # --------------------------------------------------------
    # Load
    # --------------------------------------------------------

    ch1 = load_tektronix_csv(
        ch1_file
    )

    ch2 = load_tektronix_csv(
        ch2_file
    )


    (
        time_s,
        total_voltage_v,
        raw_sense_voltage_v

    ) = align_channels(
        ch1,
        ch2
    )


    # --------------------------------------------------------
    # Calculate voltage/current
    # --------------------------------------------------------

    (

        corrected_sense_raw_v,

        corrected_sense_filtered_v,

        cell_voltage_v,

        current_raw_a,

        current_filtered_a,

        current_density_raw_mA_cm2,

        current_density_filtered_mA_cm2

    ) = calculate_cv_arrays(

        time_s,

        total_voltage_v,

        raw_sense_voltage_v,

        area_cm2
    )


    # --------------------------------------------------------
    # Detect and analyze cycles
    #
    # FILTERED current is used for CSC.
    # --------------------------------------------------------

    (
        cycle_df,
        retained_cycles

    ) = analyze_cycles(

        time_s,

        cell_voltage_v,

        current_filtered_a,

        area_cm2
    )


    # --------------------------------------------------------
    # Add metadata
    # --------------------------------------------------------

    cycle_df["Wafer"] = (
        info["Wafer"]
    )

    cycle_df["Device"] = (
        info["Device"]
    )

    cycle_df["Material"] = (
        info["Material"]
    )

    cycle_df["Size_um"] = (
        size_um
    )

    cycle_df["Area_um2"] = (
        area_um2
    )


    # --------------------------------------------------------
    # Save cycle results
    # --------------------------------------------------------

    cycle_df.to_csv(

        folder /
        f"{folder.name}_CV_cycles.csv",

        index=False
    )


    # --------------------------------------------------------
    # Save full waveform
    # --------------------------------------------------------

    waveform_df = pd.DataFrame({

        "Time_s":
            time_s,

        "Total_Voltage_V":
            total_voltage_v,

        "Sense_Voltage_Raw_V":
            raw_sense_voltage_v,

        "Sense_Voltage_Offset_Corrected_V":
            corrected_sense_raw_v,

        "Sense_Voltage_Filtered_V":
            corrected_sense_filtered_v,

        "Cell_Voltage_V":
            cell_voltage_v,

        "Current_Raw_A":
            current_raw_a,

        "Current_Raw_uA":
            current_raw_a * 1e6,

        "Current_Filtered_A":
            current_filtered_a,

        "Current_Filtered_uA":
            current_filtered_a * 1e6,

        "Current_Density_Raw_mA_cm2":
            current_density_raw_mA_cm2,

        "Current_Density_Filtered_mA_cm2":
            current_density_filtered_mA_cm2
    })


    waveform_df.to_csv(

        folder /
        f"{folder.name}_CV_processed.csv",

        index=False
    )


    # ========================================================
    # PLOT 1:
    # Individual retained cycles overlaid
    # ========================================================

    plot_overlaid_cycles(

        folder,

        time_s,

        cell_voltage_v,

        current_density_filtered_mA_cm2,

        retained_cycles
    )


    # ========================================================
    # PLOT 2:
    # Mean CV +/- SD
    # ========================================================

    plot_mean_cv(

        folder,

        cell_voltage_v,

        current_density_filtered_mA_cm2,

        retained_cycles
    )


    # --------------------------------------------------------
    # Summary statistics
    # --------------------------------------------------------

    n_cycles = len(
        cycle_df
    )


    cathodic_mean = cycle_df[
        "Cathodic_CSC_mC_cm2"
    ].mean()


    anodic_mean = cycle_df[
        "Anodic_CSC_mC_cm2"
    ].mean()


    if n_cycles > 1:

        cathodic_sd = cycle_df[
            "Cathodic_CSC_mC_cm2"
        ].std(ddof=1)


        anodic_sd = cycle_df[
            "Anodic_CSC_mC_cm2"
        ].std(ddof=1)

    else:

        cathodic_sd = np.nan

        anodic_sd = np.nan


    print()

    print(
        f"  Cycles analyzed: "
        f"{n_cycles}"
    )


    print(
        f"  Cathodic CSC: "
        f"{cathodic_mean:.3f} "
        f"+/- {cathodic_sd:.3f} mC/cm²"
    )


    print(
        f"  Anodic CSC: "
        f"{anodic_mean:.3f} "
        f"+/- {anodic_sd:.3f} mC/cm²"
    )


    print(
        f"  Low-pass filter: "
        f"{'ON' if apply_lowpass_filter else 'OFF'}"
    )


    if apply_lowpass_filter:

        print(
            f"  LPF cutoff: "
            f"{lowpass_cutoff_hz:g} Hz"
        )


    return {

        "Electrode":
            folder.name,

        "Wafer":
            info["Wafer"],

        "Device":
            info["Device"],

        "Material":
            info["Material"],

        "Size_um":
            size_um,

        "Area_um2":
            area_um2,

        "Cycles_Analyzed":
            n_cycles,

        "Discarded_First_Cycles":
            discard_first_n_cycles,

        "Ch2_Offset_Applied_mV":
            (
                ch2_offset_mV
                if apply_ch2_offset_correction
                else 0
            ),

        "Lowpass_Filter_Applied":
            apply_lowpass_filter,

        "Lowpass_Cutoff_Hz":
            (
                lowpass_cutoff_hz
                if apply_lowpass_filter
                else np.nan
            ),

        "Cathodic_CSC_Mean_mC_cm2":
            cathodic_mean,

        "Cathodic_CSC_SD_mC_cm2":
            cathodic_sd,

        "Anodic_CSC_Mean_mC_cm2":
            anodic_mean,

        "Anodic_CSC_SD_mC_cm2":
            anodic_sd
    }


# ============================================================
# PROCESS ALL ELECTRODES
# ============================================================

summary_rows = []


for folder in root_folder.iterdir():

    if not folder.is_dir():
        continue


    try:

        summary = process_electrode(
            folder
        )

        summary_rows.append(
            summary
        )


    except Exception as e:

        print()

        print(
            f"ERROR processing "
            f"{folder.name}: {e}"
        )


# ============================================================
# SAVE MASTER SUMMARY
# ============================================================

summary_df = pd.DataFrame(
    summary_rows
)


summary_file = (
    root_folder /
    "CV_summary.csv"
)


summary_df.to_csv(
    summary_file,
    index=False
)


print()
print("=" * 60)
print("CV PROCESSING COMPLETE")
print("=" * 60)

print(
    f"Summary saved to:\n"
    f"{summary_file}"
)

{'Record Length': 5000.0, 'Sample Interval': 1e-05, 'Trigger Point': 2805.0, 'Trigger Time': 5.16113586e-06, 'Horizontal Offset': -0.02805}
-0.179999999
